# Zero-Shot Antimalarial Generation — 100 000 Unique Molecules
**Pipeline:** LLM generation → Lipinski Ro5 → SA Score (≤ 4.5) → Tanimoto Novelty (max sim ≤ 0.6 vs seen) → AMCS + Lipinski descriptor filter  
**Scale:** 1 000 molecules / iteration × 100 iterations → 100 000 unique molecules


In [ ]:
# Install once per session
!pip install openai rdkit pandas --quiet

# Install openchemlib via npm
import subprocess
subprocess.run(['npm', 'install', 'openchemlib', '--prefix', '/tmp/ocl'],
               check=True, capture_output=True)
print('openchemlib installed.')

import os, re, json, textwrap
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
try:
    from rdkit.Chem import RDConfig
    import sys
    sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
    import sascorer
    _SA_FROM_CONTRIB = True
except Exception:
    _SA_FROM_CONTRIB = False

# Fallback SA scorer using standalone sascorer (pip install sascorer)
if not _SA_FROM_CONTRIB:
    try:
        import sascorer                       # pip install sascorer
        _SA_FROM_CONTRIB = True
    except ImportError:
        pass

from openai import OpenAI
import getpass

openrouter_api_key = getpass.getpass('Paste your OpenRouter API key and press Enter: ')
client = OpenAI(api_key=openrouter_api_key, base_url='https://openrouter.ai/api/v1')
MODEL = 'anthropic/claude-3-5-haiku'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.9/353.9 kB 31.5 MB/s eta 0:00:00
openchemlib installed.
Paste your OpenRouter API key and press Enter: ··········


In [ ]:
def call_llm(system_prompt, user_prompt, model=MODEL, temperature=0.9):
    response = client.chat.completions.create(
        model=model, temperature=temperature,
        messages=[{'role': 'system', 'content': system_prompt},
                  {'role': 'user',   'content': user_prompt}]
    )
    return response.choices[0].message.content

def extract_valid_smiles(llm_response_text):
    candidates = re.findall(r'[A-Za-z][A-Za-z0-9@+\-\[\]\(\)=#$\/\\%\.]+',
                            llm_response_text)
    common_words = {'The','This','These','Here','Note','Each','For',
                    'With','From','SMILES','InChI'}
    valid = []
    for c in candidates:
        if len(c) < 3 or c in common_words: continue
        if not re.search(r'[0-9@+\-\[\]\(\)=#$\/\\%\.]', c): continue
        mol = Chem.MolFromSmiles(c)
        if mol: valid.append(Chem.MolToSmiles(mol))
    seen, uniq = set(), []
    for s in valid:
        if s not in seen: seen.add(s); uniq.append(s)
    return uniq


In [ ]:
QUINOLINE_SMARTS   = 'c1ccc2ncccc2c1'
QUINAZOLINE_SMARTS = 'c1ccc2ncncc2c1'
FORBIDDEN_PATTERNS = [Chem.MolFromSmarts(s)
                      for s in (QUINOLINE_SMARTS, QUINAZOLINE_SMARTS)]

def veto_forbidden_scaffolds(smiles_list):
    kept = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None: continue
        if any(mol.HasSubstructMatch(p) for p in FORBIDDEN_PATTERNS): continue
        kept.append(smi)
    return kept


In [ ]:
import sascorer
import os
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

# ── SA Score Filter ──────────────────────────────────────────────────────
# Requires RDKit SA_Score contrib or the 'sascorer' PyPI package.
# Molecules with SA score > 4.5 are considered too hard to synthesise and discarded.

SA_THRESHOLD = 4.5

def _get_sa_score(mol):
    """Return SA score for an RDKit mol object, or None on failure."""
    try:
        if sascorer:
            return sascorer.calculateScore(mol)
        # Manual lightweight SA score approximation when sascorer is unavailable.
        # Based on ring complexity + stereo + fraction sp3 + rotatable bonds.
        # NOTE: this is an approximation; install sascorer for exact values.

        n_rings  = rdMolDescriptors.CalcNumRings(mol)
        n_stereo = len(Chem.FindMolChiralCenters(mol, includeUnassigned=True))
        fsp3     = rdMolDescriptors.CalcFractionCSP3(mol)
        score    = 1.0 + 0.5*n_stereo + 0.3*max(0, n_rings-2) + (1-fsp3)*0.5
        return min(score, 6.0)
    except Exception:
        return None

def filter_sa_score(smiles_list, threshold=SA_THRESHOLD):
    """Keep molecules with SA score ≤ threshold. Discard rest."""
    kept, discarded = [], 0
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None: discarded += 1; continue
        sa = _get_sa_score(mol)
        if sa is None or sa > threshold:
            discarded += 1
        else:
            kept.append(smi)
    print(f'[SA Filter] Kept {len(kept)}, discarded {discarded} (SA > {threshold})')
    return kept

In [ ]:
# ── Tanimoto Novelty Filter ──────────────────────────────────────────────
# Uses Morgan fingerprints (radius=2, 2048 bits) via RDKit.
# A new molecule is discarded if its maximum Tanimoto similarity
# to ANY already-accepted molecule exceeds SIM_THRESHOLD.

from rdkit.Chem import AllChem, DataStructs
import numpy as np

SIM_THRESHOLD = 0.6   # matches diagram: discard if max sim > 0.6

def _morgan_fp(smi, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)

def build_fp_library(smiles_set):
    """Pre-compute fingerprints for a set of SMILES strings."""
    fps = []
    for smi in smiles_set:
        fp = _morgan_fp(smi)
        if fp is not None:
            fps.append(fp)
    return fps

def filter_tanimoto_novelty(candidate_smiles, ref_fps, threshold=SIM_THRESHOLD):
    """
    Keep candidates whose max Tanimoto similarity to ref_fps is ≤ threshold.
    If ref_fps is empty, all candidates pass.
    Returns (kept_smiles, updated_ref_fps).
    """
    if not ref_fps:
        # No reference yet — accept all and add their fps
        new_fps = []
        kept = []
        for smi in candidate_smiles:
            fp = _morgan_fp(smi)
            if fp is not None:
                kept.append(smi)
                new_fps.append(fp)
        return kept, ref_fps + new_fps

    kept, new_fps, discarded = [], [], 0
    for smi in candidate_smiles:
        fp = _morgan_fp(smi)
        if fp is None:
            discarded += 1
            continue
        sims = DataStructs.BulkTanimotoSimilarity(fp, ref_fps)
        if max(sims) > threshold:
            discarded += 1
        else:
            kept.append(smi)
            new_fps.append(fp)
            ref_fps.append(fp)   # update library on-the-fly

    print(f'[Tanimoto] Kept {len(kept)}, discarded {discarded} '
          f'(max sim > {threshold}; ref library = {len(ref_fps)} fps)')
    return kept, ref_fps


In [ ]:
# ── OpenChemLib-JS descriptor calculation ──
import subprocess, json

_OCL_LIB = '/tmp/ocl/node_modules/openchemlib/dist/openchemlib.js'

_OCL_JS_SCRIPT = r"""
import OCL from '""" + "'" + r""" + _OCL_LIB + r""" + "'" + r""";
const { Molecule, MoleculeProperties } = OCL;
"""

# (full JS script kept exactly as in original — only pasted here for reference)
_OCL_JS_SCRIPT = r"""
import OCL from '/tmp/ocl/node_modules/openchemlib/dist/openchemlib.js';
const { Molecule, MoleculeProperties } = OCL;

function getFakeOxoCount(mol, atom) {
  let n = 0;
  for (let i = 0; i < mol.getConnAtoms(atom); i++) {
    const no = mol.getAtomicNo(mol.getConnAtom(atom, i));
    if (mol.getConnBondOrder(atom, i) === 2 && (no===7||no===8||no===16)) n++;
  }
  return n;
}

function isStabilized(mol, atom, twice) {
  let found = false;
  for (let i = 0; i < mol.getConnAtoms(atom); i++) {
    const bond = mol.getConnBond(atom, i);
    if (!mol.isAromaticBond(bond) && mol.getConnBondOrder(atom, i) === 1) {
      const conn = mol.getConnAtom(atom, i);
      if (!mol.isAromaticAtom(conn) &&
          ((mol.getAtomicNo(conn)===6 && getFakeOxoCount(mol,conn)===1) ||
           (mol.getAtomicNo(conn)===16 && getFakeOxoCount(mol,conn)===2))) {
        if (found || !twice) return true;
        found = true;
      }
    }
  }
  return false;
}

function isVinylogFakeOxo(mol, atom) {
  for (let i = 0; i < mol.getConnAtoms(atom); i++) {
    if (mol.getConnBondOrder(atom, i) !== 1) {
      const conn = mol.getConnAtom(atom, i);
      for (let j = 0; j < mol.getConnAtoms(conn); j++)
        if (mol.getConnBondOrder(conn,j)===1 && getFakeOxoCount(mol,mol.getConnAtom(conn,j))!==0)
          return true;
    }
  }
  return false;
}

function isBasicNitrogen(mol, atom) {
  if (mol.getAtomicNo(atom) !== 7) return false;
  if (mol.getAtomCharge(atom) !== 0) return false;
  if (mol.getConnAtoms(atom) + mol.getAtomPi(atom) > 3) return false;
  if (mol.isAromaticAtom(atom)) {
    if (mol.getAtomPi(atom) !== 1) return false;
    if (mol.getAtomRingCount(atom, 7) !== 1) return false;
    const rc = mol.getRingSet();
    for (let r = 0; r < rc.getSize(); r++) {
      if (!rc.isAtomMember(r, atom)) continue;
      const sz = rc.getRingSize(r);
      if (sz !== 5 && sz !== 6) { break; }
      const ring = rc.getRingAtoms(r);
      const R = Array.from({length:sz}, (_,i)=>ring[i]);
      let ni = R.indexOf(atom);
      let opi, mi;
      if (sz === 5) {
        opi = [R[ni-1<0?ni+4:ni-1], R[ni-4<0?ni+1:ni-4]];
        mi  = [R[ni-2<0?ni+3:ni-2], R[ni-3<0?ni+2:ni-3]];
      } else {
        opi = [R[ni-1<0?ni+5:ni-1], R[ni-3<0?ni+3:ni-3], R[ni-5<0?ni+1:ni-5]];
        mi  = [R[ni-2<0?ni+4:ni-2], R[ni-4<0?ni+2:ni-4]];
      }
      let ec = 0;
      for (let i=0; i<R.length; i++)
        if (atom!==R[i] && mol.getAtomicNo(R[i])===7 && mol.getAtomPi(R[i])===1) ec--;
      for (let i=0; i<opi.length; i++) {
        let exoAtom=-1, exoBond=-1;
        for (let j=0; j<mol.getConnAtoms(opi[i]); j++) {
          if (!mol.isAromaticBond(mol.getConnBond(opi[i],j))) {
            exoAtom=mol.getConnAtom(opi[i],j); exoBond=mol.getConnBond(opi[i],j); break;
          }
        }
        if (exoAtom !== -1) {
          if (mol.getAtomicNo(exoAtom)===7 && mol.getAtomPi(exoAtom)===0
              && (mol.getConnAtoms(exoAtom)+mol.getAtomPi(exoAtom)<=3)
              && !isStabilized(mol,exoAtom,false)) { ec++; continue; }
          if (mol.getAtomicNo(exoAtom)===8 && mol.getConnAtoms(exoAtom)===1) { ec+=2; continue; }
          if (mol.isAromaticBond(exoBond)) {
            for (let s=0; s<rc.getSize(); s++) {
              if (rc.isAromatic(s) && rc.isAtomMember(s,exoAtom)) {
                const ra=rc.getRingAtoms(s);
                for (let j=0; j<rc.getRingSize(s); j++)
                  if (mol.getAtomicNo(ra[j])===7 && mol.getAtomPi(ra[j])===1) { ec--; break; }
                break;
              }
            }
          }
        }
      }
      for (let i=0; i<mi.length; i++) {
        let exoAtom=-1;
        for (let j=0; j<mol.getConnAtoms(mi[i]); j++)
          if (!mol.isAromaticBond(mol.getConnBond(mi[i],j))) exoAtom=mol.getConnAtom(mi[i],j);
        if (mol.getAtomicNo(mi[i])===6) {
          if (exoAtom!==-1 && getFakeOxoCount(mol,exoAtom)!==0) ec--;
        } else if (mol.getAtomicNo(mi[i])===7) {
          if (mol.getAtomPi(mi[i])===0 &&
              (exoAtom===-1 || (!mol.isAromaticAtom(exoAtom) && getFakeOxoCount(mol,exoAtom)===0)))
            ec++;
        }
      }
      return ec > 0;
    }
    return false;
  }
  if (mol.getAtomPi(atom) > 1) return false;
  if (mol.getAtomPi(atom) === 1) {
    let imineC=-1, sc=0;
    for (let i=0; i<mol.getConnAtoms(atom); i++) {
      const conn=mol.getConnAtom(atom,i);
      if (mol.getConnBondOrder(atom,i)===2) {
        if (mol.getAtomicNo(conn)!==6) return false;
        imineC=conn; continue;
      }
      if (mol.getAtomicNo(conn)===8) return false;
      if (mol.getAtomicNo(conn)===7) { sc--; if (isStabilized(mol,conn,false)) sc--; continue; }
      if (mol.isAromaticAtom(conn)) sc--;
    }
    if (imineC===-1) return false;
    let arNC=0;
    for (let i=0; i<mol.getConnAtoms(imineC); i++) {
      if (mol.getConnBondOrder(imineC,i)===1) {
        const conn=mol.getConnAtom(imineC,i);
        if (getFakeOxoCount(mol,conn)!==0) return false;
        if (mol.isAromaticAtom(conn)) arNC++;
        if (mol.getAtomicNo(conn)===7 && !isStabilized(mol,conn,true)) sc++;
        if (mol.getAtomicNo(conn)===8||mol.getAtomicNo(conn)===16) sc--;
      }
    }
    if (arNC===2) sc--;
    return sc >= 0;
  }
  for (let i=0; i<mol.getConnAtoms(atom); i++) {
    const conn=mol.getConnAtom(atom,i);
    if (mol.isAromaticAtom(conn)) return false;
    if (mol.getAtomicNo(conn)!==6) return false;
    if (getFakeOxoCount(mol,conn)!==0) return false;
    if (mol.getAtomPi(conn)!==0 && isVinylogFakeOxo(mol,conn)) return false;
  }
  return true;
}

function countBasicNitrogens(mol) {
  let n=0;
  for (let i=0; i<mol.getAllAtoms(); i++) if (isBasicNitrogen(mol,i)) n++;
  return n;
}

const smilesList = JSON.parse(process.argv[2]);
const results = [];
for (const smi of smilesList) {
  try {
    const mol = Molecule.fromSmiles(smi);
    mol.ensureHelperArrays(255);
    const props = new MoleculeProperties(mol);
    const formula = mol.getMolecularFormula();
    results.push({
      Structure:              smi,
      cLogP:                  parseFloat(props.logP.toFixed(3)),
      molecularWeight:        parseFloat(formula.relativeWeight.toFixed(3)),
      topologicalSurfaceArea: parseFloat(props.polarSurfaceArea.toFixed(3)),
      hDonorCount:            props.donorCount,
      hAcceptorCount:         props.acceptorCount,
      rotatableBondCount:     props.rotatableBondCount,
      basicNitrogenCount:     countBasicNitrogens(mol),
      aromaticRingCount:      mol.getAromaticRingCount(),
    });
  } catch(e) {
    results.push({Structure: smi, error: e.message});
  }
}
process.stdout.write(JSON.stringify(results));
"""

DW_COL = {
    'basic_n': 'basicNitrogenCount',
    'arom_r':  'aromaticRingCount',
    'clogp':   'cLogP',
    'tpsa':    'topologicalSurfaceArea',
    'hba':     'hAcceptorCount',
    'hbd':     'hDonorCount',
    'mw':      'molecularWeight',
    'rotb':    'rotatableBondCount',
}

def run_ocl_calc(smiles_list: list) -> list:
    if not smiles_list:
        print('[OCL] No molecules to process.')
        return []
    with open('/tmp/_ocl_calc.mjs', 'w') as f:
        f.write(_OCL_JS_SCRIPT)
    result = subprocess.run(
        ['node', '/tmp/_ocl_calc.mjs', json.dumps(smiles_list)],
        capture_output=True, text=True, timeout=300
    )
    if result.returncode != 0:
        print(f'[OCL] Node.js error: {result.stderr[:400]}')
        return []
    rows = json.loads(result.stdout)
    valid = [r for r in rows if 'error' not in r]
    if len(rows) - len(valid):
        print(f'[OCL] Skipped {len(rows)-len(valid)} unparseable SMILES.')
    print(f'[OCL] Descriptors calculated for {len(valid)} molecules.')
    return valid

def filter_by_rules(rows: list, min_amcs: int = 3, min_lipinski: int = 4) -> list:
    def _f(row, key, fallback=0.0):
        try: return float(row.get(DW_COL[key], fallback))
        except (ValueError, TypeError): return fallback
    filtered = []
    for row in rows:
        amcs = sum([_f(row,'basic_n')>=1, _f(row,'arom_r')>=2,
                    _f(row,'clogp')>=2, _f(row,'tpsa')<=80])
        lip  = sum([_f(row,'hba')<=10, _f(row,'hbd')<=5,
                    _f(row,'clogp')<=5, _f(row,'mw')<=500, _f(row,'rotb')<=10])
        if amcs >= min_amcs and lip >= min_lipinski:
            row['AMCS_passes'] = amcs
            row['Lipinski_passes'] = lip
            filtered.append(row)
    return filtered

print('OpenChemLib-JS descriptor calc + Lipinski filter loaded.')


OpenChemLib-JS descriptor calc + Lipinski filter loaded.


In [ ]:
def generate_zero_shot_adv(n_generate=1000, overgenerate_factor=3,
                            seen_smiles=None, ref_fps=None):
    """
    Generate n_generate novel antimalarial SMILES per call.

    Pipeline per iteration:
      LLM → RDKit parse → scaffold veto → SA score filter (≤ 4.5)
      → Tanimoto novelty filter (max sim ≤ 0.6 vs seen set)
      → dedup vs seen_smiles → OCL descriptor calc → AMCS+Lipinski filter
      → return up to n_generate rows

    Parameters
    ----------
    n_generate : int
        Target number of molecules to return after all filters.
    overgenerate_factor : int
        Ask the LLM for n_generate * overgenerate_factor candidates to
        compensate for filter attrition.
    seen_smiles : set
        Canonical SMILES accepted in all previous iterations (dedup guard).
    ref_fps : list
        Morgan fingerprints of all previously accepted molecules
        (Tanimoto novelty guard).

    Returns
    -------
    filtered_rows : list[dict]   descriptor rows for accepted molecules
    smiles_final  : list[str]    canonical SMILES of accepted molecules
    raw           : str          raw LLM response text
    ref_fps       : list         updated fingerprint library
    """
    seen_smiles = seen_smiles or set()
    ref_fps     = ref_fps     if ref_fps is not None else []

    system_prompt = """You are a computational medicinal chemist.
Your job is to propose novel antimalarial drug candidates in SMILES format.
Hard constraints:
- DO NOT output any molecule whose core scaffold is quinoline or quinazoline (or obvious fused variants).
- Maintain structural diversity; avoid close analogs among your own proposals.
Output rules: SMILES only, one per line, no names, no numbering, no commentary."""

    exclusion_block = ''
    if seen_smiles:
        sample = list(seen_smiles)[-200:]
        exclusion_block = (
            f'\nDo NOT reproduce any of the following {len(sample)} already-generated SMILES '
            f'(or any molecule that is identical after canonicalisation):\n'
            + '\n'.join(sample) + '\n'
        )

    user_prompt = (
        f'Generate {n_generate * overgenerate_factor} NEW, NOVEL antimalarial candidate SMILES.\n'
        f'Respect the scaffold ban above.{exclusion_block}'
        f'Return SMILES only, one per line.'
    )

    # ── Step 1: LLM generation ────────────────────────────────────────────
    print('[LLM] Calling model...')
    raw = call_llm(system_prompt, user_prompt)
    parsed = extract_valid_smiles(raw)
    print(f'[LLM] Parsed {len(parsed)} valid SMILES.')

    # ── Step 2: Scaffold veto ─────────────────────────────────────────────
    parsed = veto_forbidden_scaffolds(parsed)
    print(f'[Scaffold] {len(parsed)} remain after quinoline/quinazoline veto.')

    # ── Step 3: Lipinski Ro5 (MW≤500 cLogP≤5 HBD≤5 HBA≤10) ──────────────
    # Quick RDKit pre-screen before expensive filters
    def _lipinski_rdkit(smi):
        mol = Chem.MolFromSmiles(smi)
        if mol is None: return False
        mw   = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        hbd  = rdMolDescriptors.CalcNumHBD(mol)
        hba  = rdMolDescriptors.CalcNumHBA(mol)
        return mw <= 500 and logp <= 5 and hbd <= 5 and hba <= 10

    before = len(parsed)
    parsed = [s for s in parsed if _lipinski_rdkit(s)]
    print(f'[Ro5] {len(parsed)} remain after Lipinski pre-screen '
          f'(removed {before - len(parsed)}).')

    # ── Step 4: SA Score filter (≤ 4.5) ──────────────────────────────────
    parsed = filter_sa_score(parsed, threshold=SA_THRESHOLD)

    # ── Step 5: Tanimoto novelty filter (max sim ≤ 0.6 vs seen) ──────────
    parsed, ref_fps = filter_tanimoto_novelty(parsed, ref_fps, threshold=SIM_THRESHOLD)

    # ── Step 6: Cross-iteration SMILES dedup ─────────────────────────────
    before_dedup = len(parsed)
    parsed = [s for s in parsed if s not in seen_smiles]
    dupes = before_dedup - len(parsed)
    if dupes:
        print(f'[Dedup] Removed {dupes} SMILES already in seen set.')
    parsed = list(dict.fromkeys(parsed))   # within-batch dedup
    print(f'[Dedup] {len(parsed)} unique new molecules after all filters.')

    if not parsed:
        return [], [], raw, ref_fps

    # ── Step 7: OCL descriptor calc + AMCS/Lipinski filter ───────────────
    print('[OCL] Calculating descriptors...')
    rows = run_ocl_calc(parsed)
    filtered_rows = filter_by_rules(rows, min_amcs=3, min_lipinski=4)[:n_generate]
    smiles_final  = [r['Structure'] for r in filtered_rows]
    print(f'[OCL] {len(smiles_final)} passed AMCS + Lipinski filter.')

    return filtered_rows, smiles_final, raw, ref_fps


In [ ]:
import os

# ── Configuration ─────────────────────────────────────────────────────────
N_ITERATIONS        = 100          # number of LLM calls
N_PER_ITER          = 1000         # target accepted molecules per iteration
OVERGENERATE_FACTOR = 4            # ask LLM for 4× to compensate filter attrition
TARGET_TOTAL        = 100_000      # stop early if reached
OUTPUT_CSV          = 'generated_antimalarials_100k.csv'

# ── State ─────────────────────────────────────────────────────────────────
all_rows    = []
seen_smiles = set()   # canonical SMILES accepted in all iterations
ref_fps     = []      # Morgan FPs for Tanimoto novelty guard

for iteration in range(1, N_ITERATIONS + 1):
    print(f'\n{"="*65}')
    print(f'Iteration {iteration}/{N_ITERATIONS}  '
          f'(unique molecules so far: {len(seen_smiles)} / {TARGET_TOTAL})')
    print(f'{"="*65}')

    # Early-stop guard
    if len(seen_smiles) >= TARGET_TOTAL:
        print(f'[DONE] Target of {TARGET_TOTAL:,} unique molecules reached. Stopping.')
        break

    rows, smiles_final, raw, ref_fps = generate_zero_shot_adv(
        n_generate=N_PER_ITER,
        overgenerate_factor=OVERGENERATE_FACTOR,
        seen_smiles=seen_smiles,
        ref_fps=ref_fps,
    )

    # Register newly accepted molecules
    for smi in smiles_final:
        seen_smiles.add(smi)

    # Label with iteration number
    for row in rows:
        row['iteration'] = iteration

    all_rows.extend(rows)
    print(f'[Iter {iteration}] {len(rows)} new molecules collected '
          f'(total so far: {len(all_rows):,})')

    # Save / append to CSV after every iteration (crash-safe)
    df_iter      = pd.DataFrame(rows)
    write_header = not os.path.exists(OUTPUT_CSV) or iteration == 1
    df_iter.to_csv(OUTPUT_CSV,
                   mode='a' if not write_header else 'w',
                   header=write_header, index=False)
    print(f'[CSV] Saved iteration {iteration} → {OUTPUT_CSV}')

    # Save raw LLM response
    with open(f'raw_llm_iter_{iteration:03d}.txt', 'w') as f:
        f.write(raw)

# ── Final summary ─────────────────────────────────────────────────────────
df_all = pd.DataFrame(all_rows)
print(f'\n{"="*65}')
print(f'DONE — {len(df_all):,} unique molecules across '
      f'{min(iteration, N_ITERATIONS)} iterations')
print(f'CSV saved to: {OUTPUT_CSV}')
print(f'Filter settings used:')
print(f'  SA Score threshold   : ≤ {SA_THRESHOLD}')
print(f'  Tanimoto threshold   : max sim ≤ {SIM_THRESHOLD} (Morgan r=2, 2048 bits)')
print(f'  Lipinski Ro5         : MW≤500, cLogP≤5, HBD≤5, HBA≤10')
print(f'  AMCS passes required : ≥ 3')
print(f'  Lipinski passes req. : ≥ 4')
print('\nDescriptor table (first 20 rows):')
display(df_all.head(20))



Iteration 1/100  (unique molecules so far: 0 / 100000)
[LLM] Calling model...


[08:27:55] SMILES Parse Error: syntax error while parsing: quinoline/quinazoline
[08:27:55] SMILES Parse Error: check for mistakes around position 1:
[08:27:55] quinoline/quinazoline
[08:27:55] ^
[08:27:55] SMILES Parse Error: Failed parsing SMILES 'quinoline/quinazoline' for input: 'quinoline/quinazoline'
[08:27:55] SMILES Parse Error: syntax error while parsing: scaffolds.
[08:27:55] SMILES Parse Error: check for mistakes around position 3:
[08:27:55] scaffolds.
[08:27:55] ~~^
[08:27:55] SMILES Parse Error: Failed parsing SMILES 'scaffolds.' for input: 'scaffolds.'
[08:27:55] SMILES Parse Error: syntax error while parsing: SMILES.
[08:27:55] SMILES Parse Error: check for mistakes around position 2:
[08:27:55] SMILES.
[08:27:55] ~^
[08:27:55] SMILES Parse Error: Failed parsing SMILES 'SMILES.' for input: 'SMILES.'
[08:27:55] DEPRECATION WARNING: please use MorganGenerator
[08:27:55] DEPRECATION WARNING: please use MorganGenerator
[08:27:55] DEPRECATION WARNING: please use MorganGenera

[LLM] Parsed 10 valid SMILES.
[Scaffold] 9 remain after quinoline/quinazoline veto.
[Ro5] 9 remain after Lipinski pre-screen (removed 0).
[SA Filter] Kept 9, discarded 0 (SA > 4.5)
[Dedup] 9 unique new molecules after all filters.
[OCL] Calculating descriptors...
[OCL] Descriptors calculated for 9 molecules.
[OCL] 1 passed AMCS + Lipinski filter.
[Iter 1] 1 new molecules collected (total so far: 1)
[CSV] Saved iteration 1 → generated_antimalarials_100k.csv

Iteration 2/100  (unique molecules so far: 1 / 100000)
[LLM] Calling model...


[08:28:00] SMILES Parse Error: syntax error while parsing: task.
[08:28:00] SMILES Parse Error: check for mistakes around position 1:
[08:28:00] task.
[08:28:00] ^
[08:28:00] SMILES Parse Error: Failed parsing SMILES 'task.' for input: 'task.'
[08:28:00] SMILES Parse Error: syntax error while parsing: novelty/diversity
[08:28:00] SMILES Parse Error: check for mistakes around position 3:
[08:28:00] novelty/diversity
[08:28:00] ~~^
[08:28:00] SMILES Parse Error: Failed parsing SMILES 'novelty/diversity' for input: 'novelty/diversity'
[08:28:00] SMILES Parse Error: syntax error while parsing: validation.
[08:28:00] SMILES Parse Error: check for mistakes around position 1:
[08:28:00] validation.
[08:28:00] ^
[08:28:00] SMILES Parse Error: Failed parsing SMILES 'validation.' for input: 'validation.'
[08:28:00] SMILES Parse Error: syntax error while parsing: high-quality
[08:28:00] SMILES Parse Error: check for mistakes around position 1:
[08:28:00] high-quality
[08:28:00] ^
[08:28:00] SMILE

[LLM] Parsed 0 valid SMILES.
[Scaffold] 0 remain after quinoline/quinazoline veto.
[Ro5] 0 remain after Lipinski pre-screen (removed 0).
[SA Filter] Kept 0, discarded 0 (SA > 4.5)
[Tanimoto] Kept 0, discarded 0 (max sim > 0.6; ref library = 9 fps)
[Dedup] 0 unique new molecules after all filters.
[Iter 2] 0 new molecules collected (total so far: 1)
[CSV] Saved iteration 2 → generated_antimalarials_100k.csv

Iteration 3/100  (unique molecules so far: 1 / 100000)
[LLM] Calling model...


...

[LLM] Parsed 0 valid SMILES.
[Scaffold] 0 remain after quinoline/quinazoline veto.
[Ro5] 0 remain after Lipinski pre-screen (removed 0).
[SA Filter] Kept 0, discarded 0 (SA > 4.5)
[Tanimoto] Kept 0, discarded 0 (max sim > 0.6; ref library = 245 fps)
[Dedup] 0 unique new molecules after all filters.
[Iter 98] 0 new molecules collected (total so far: 171)
[CSV] Saved iteration 98 → generated_antimalarials_100k.csv

Iteration 99/100  (unique molecules so far: 171 / 100000)
[LLM] Calling model...


[08:38:14] SMILES Parse Error: syntax error while parsing: understand.
[08:38:14] SMILES Parse Error: check for mistakes around position 1:
[08:38:14] understand.
[08:38:14] ^
[08:38:14] SMILES Parse Error: Failed parsing SMILES 'understand.' for input: 'understand.'
[08:38:14] SMILES Parse Error: syntax error while parsing: quinoline/quinazoline
[08:38:14] SMILES Parse Error: check for mistakes around position 1:
[08:38:14] quinoline/quinazoline
[08:38:14] ^
[08:38:14] SMILES Parse Error: Failed parsing SMILES 'quinoline/quinazoline' for input: 'quinoline/quinazoline'
[08:38:14] SMILES Parse Error: syntax error while parsing: SMILES.
[08:38:14] SMILES Parse Error: check for mistakes around position 2:
[08:38:14] SMILES.
[08:38:14] ~^
[08:38:14] SMILES Parse Error: Failed parsing SMILES 'SMILES.' for input: 'SMILES.'
[08:38:14] SMILES Parse Error: syntax error while parsing: diversity.
[08:38:14] SMILES Parse Error: check for mistakes around position 1:
[08:38:14] diversity.
[08:38:14]

[LLM] Parsed 0 valid SMILES.
[Scaffold] 0 remain after quinoline/quinazoline veto.
[Ro5] 0 remain after Lipinski pre-screen (removed 0).
[SA Filter] Kept 0, discarded 0 (SA > 4.5)
[Tanimoto] Kept 0, discarded 0 (max sim > 0.6; ref library = 245 fps)
[Dedup] 0 unique new molecules after all filters.
[Iter 99] 0 new molecules collected (total so far: 171)
[CSV] Saved iteration 99 → generated_antimalarials_100k.csv

Iteration 100/100  (unique molecules so far: 171 / 100000)
[LLM] Calling model...
[LLM] Parsed 0 valid SMILES.
[Scaffold] 0 remain after quinoline/quinazoline veto.
[Ro5] 0 remain after Lipinski pre-screen (removed 0).
[SA Filter] Kept 0, discarded 0 (SA > 4.5)
[Tanimoto] Kept 0, discarded 0 (max sim > 0.6; ref library = 245 fps)
[Dedup] 0 unique new molecules after all filters.
[Iter 100] 0 new molecules collected (total so far: 171)
[CSV] Saved iteration 100 → generated_antimalarials_100k.csv

DONE — 171 unique molecules across 100 iterations
CSV saved to: generated_antimala

[08:38:18] SMILES Parse Error: syntax error while parsing: task.
[08:38:18] SMILES Parse Error: check for mistakes around position 1:
[08:38:18] task.
[08:38:18] ^
[08:38:18] SMILES Parse Error: Failed parsing SMILES 'task.' for input: 'task.'
[08:38:18] SMILES Parse Error: syntax error while parsing: quinoline/quinazoline
[08:38:18] SMILES Parse Error: check for mistakes around position 1:
[08:38:18] quinoline/quinazoline
[08:38:18] ^
[08:38:18] SMILES Parse Error: Failed parsing SMILES 'quinoline/quinazoline' for input: 'quinoline/quinazoline'
[08:38:18] SMILES Parse Error: syntax error while parsing: molecules.
[08:38:18] SMILES Parse Error: check for mistakes around position 1:
[08:38:18] molecules.
[08:38:18] ^
[08:38:18] SMILES Parse Error: Failed parsing SMILES 'molecules.' for input: 'molecules.'
[08:38:18] SMILES Parse Error: syntax error while parsing: quinoline/quinazoline
[08:38:18] SMILES Parse Error: check for mistakes around position 1:
[08:38:18] quinoline/quinazoline
[

,Structure,cLogP,molecularWeight,topologicalSurfaceArea,hDonorCount,hAcceptorCount,rotatableBondCount,basicNitrogenCount,aromaticRingCount,AMCS_passes,Lipinski_passes,iteration
0,Cc1onc(-c2ccccc2)c1C(=O)NC1CC1,2.187,242.277,55.13,1,4,3,0,2,3,5,1
1,CC1(C)Oc2cc(NC(=O)c3cnco3)ccc21,2.161,244.249,64.36,1,5,2,0,2,3,5,6
2,O=C1NC(=Cc2ccccc2)c2ccccc21,3.167,221.258,29.10,1,2,1,0,2,3,5,6
3,Cc1onc(-c2ccc(F)cc2)c1C(=O)N1CCC(C)C1,3.084,288.321,46.34,0,4,2,0,2,3,5,6
4,O=C(c1ccccc1)N1CC(c2ccc(F)cc2)=N1,2.915,254.263,32.67,0,3,2,0,2,3,5,6
5,O=C1c2ccccc2C(=O)N1Cc1ccccc1,2.412,237.257,37.38,0,3,2,0,2,3,5,6
6,O=C(Cc1ccc(Cl)cc1)c1ccc(N2CCOCC2)cc1,3.248,315.799,29.54,0,3,4,0,2,3,5,11
7,CCOc1ccc(C(=O)Nc2ccc(C(F)(F)F)cc2)cc1,3.992,309.286,38.33,1,3,5,0,2,3,5,11
8,Cc1onc(-c2cc(Br)ccc2O)c1C(=O)NC(C)C,2.802,339.188,75.36,2,5,3,0,2,3,5,11
9,O=C(Nc1cccc(C(F)(F)F)c1)c1ccc(N2CCCC2)cc1,4.146,334.340,32.34,1,3,4,0,2,3,5,11
